# 02 — NumPy Memory and Zero-Copy Views
## Shape, strides, who owns the memory, and streaming buffers

**Audience:** generalist engineering students with varied backgrounds
**Format:** context → mental model → worked examples → checks → open project
**Rule:** every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell that puts the output in plain words.
- **`assert` lines** — the specification. If one fails after you edit a cell, your change broke a rule.

The project at the end is open: you get the goal and the acceptance criteria.

## Before we start: what a NumPy array actually is

A Python list is a box of pointers to objects. A NumPy array is different: it is **one block of numbers** in memory, plus a small label that says how to read that block.

The label has four parts:

- **`dtype`** — the type and size of each number (`int32` = 4 bytes, `float64` = 8 bytes).
- **`shape`** — how many elements along each axis, e.g. `(3, 4)` = 3 rows, 4 columns.
- **`strides`** — how many **bytes** to jump to move one step along each axis.
- **who owns the memory** — did this array allocate its own block, or is it looking at someone else's?

Two words you will see constantly:

- A **view** looks at another array's memory. Making one is free. But writing to a view writes to the original too.
- A **copy** has its own memory. Safe to change, but costs time and space to make.
- **Zero-copy** means "reuse the existing block instead of copying it". It does **not** mean the data is read-only.

The habit this notebook builds: whenever an array crosses a place where speed matters, look at `shape`, `dtype`, `strides`, whether it is contiguous, and whether it shares memory.

## 1. An array is a label over a block of bytes

`strides` counts **bytes**, not elements.

**Predict** the `strides` of `np.arange(12, dtype=np.int32).reshape(3, 4)`.
Hint: `int32` is 4 bytes. The 4 numbers of a row sit end to end, so moving one **column** jumps 4 bytes. A row is 4 columns, so moving one **row** jumps `4 × 4 = 16` bytes.

![Array memory and strides](assets/array_strides.svg)

In [1]:
import numpy as np

x = np.arange(12, dtype=np.int32).reshape(3, 4)
print(x)
print()
print("shape   :", x.shape)
print("dtype   :", x.dtype, f"({x.itemsize} bytes per element)")
print("strides :", x.strides, "  <- (bytes per row, bytes per column)")

assert x.strides == (16, 4)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

shape   : (3, 4)
dtype   : int32 (4 bytes per element)
strides : (16, 4)   <- (bytes per row, bytes per column)


### What you just saw

Same 12 numbers in memory, read as 3 rows of 4. `strides == (16, 4)` means: to go to the next column, jump 4 bytes; to go to the next row, jump 16 bytes.

`shape` and `strides` are just *instructions for reading* the block. They do not touch the numbers themselves — which is why the next operations can be free.

## 2. View vs copy

- **Basic slicing** (`x[:, 1:3]`) usually gives a **view** — same memory.
- **Advanced indexing** (`x[:, [1, 2]]`, a list of positions) usually gives a **copy**.

The values can look identical, so do not eyeball it. Use `np.shares_memory(a, b)` — it is the reliable test. (Do not use `.base`; it can be misleading.)

**Predict.** Below, `view = x[:, 1:3]` and `copy = x[:, [1, 2]]`. Which one shares memory with `x`? After `view[0, 0] = 999`, what is `x[0, 1]`?

In [2]:
view = x[:, 1:3]        # basic slice  -> view
copy = x[:, [1, 2]]     # list of indices -> copy

print("view shares memory with x:", np.shares_memory(x, view))
print("copy shares memory with x:", np.shares_memory(x, copy))

view[0, 0] = 999        # writing to the view...
print("\nx[0, 1] is now:", x[0, 1], "  <- the write went through to x")
print("x =\n", x)

assert np.shares_memory(x, view)
assert not np.shares_memory(x, copy)
assert x[0, 1] == 999

view shares memory with x: True
copy shares memory with x: False

x[0, 1] is now: 999   <- the write went through to x
x =
 [[  0 999   2   3]
 [  4   5   6   7]
 [  8   9  10  11]]


### What you just saw

`view` is a window onto `x`'s memory, so `view[0, 0] = 999` also changed `x[0, 1]` (position `[0,0]` of the sliced window is position `[0,1]` of `x`).

`copy` has its own memory; writing to it would leave `x` alone. Use a view when you want cheap access; use a copy when the result must be independent.

## 3. Transpose just relabels

`x.T` does not move any numbers. It swaps `shape` and reverses `strides`, so the same block is now read column-first. Some libraries need their input laid out row-first ("C-contiguous") and will quietly make a copy if it is not.

In [3]:
xt = x.T

print("x  shape", x.shape, " strides", x.strides, " C-contiguous:", x.flags.c_contiguous)
print("xT shape", xt.shape, " strides", xt.strides, " C-contiguous:", xt.flags.c_contiguous)
print("xT shares memory with x:", np.shares_memory(x, xt))
print("xT[2, 1] == x[1, 2]:", xt[2, 1] == x[1, 2])

assert np.shares_memory(x, xt)
assert xt[2, 1] == x[1, 2]

x  shape (3, 4)  strides (16, 4)  C-contiguous: True
xT shape (4, 3)  strides (4, 16)  C-contiguous: False
xT shares memory with x: True
xT[2, 1] == x[1, 2]: True


### What you just saw

`xt` shares `x`'s memory but its strides are swapped, so `xt.flags.c_contiguous` is `False`. That is not a bug — it is just a different reading order over the same bytes. If you hand `xt` to code that needs contiguous input, that code will copy it for you.

### Making that copy visible

`np.ascontiguousarray(a)` returns `a` unchanged if it is already row-first, or a fresh contiguous copy if it is not. Checking `shares_memory` afterwards tells you which happened — useful when that hidden copy would cost you latency or memory.

In [4]:
non_contig = x.T
contiguous = np.ascontiguousarray(non_contig)      # not row-first -> makes a copy

print("non_contig C-contiguous:", non_contig.flags.c_contiguous)
print("contiguous  C-contiguous:", contiguous.flags.c_contiguous)
print("a copy was made         :", not np.shares_memory(non_contig, contiguous))
print("same values             :", np.array_equal(non_contig, contiguous))

already_ok = np.ascontiguousarray(x)              # x is ALREADY row-first
print("x reused, no copy       :", np.shares_memory(x, already_ok))

assert not np.shares_memory(non_contig, contiguous)
assert np.shares_memory(x, already_ok)

non_contig C-contiguous: False
contiguous  C-contiguous: True
a copy was made         : True
same values             : True
x reused, no copy       : True


## 4. Sliding windows without copying

`np.lib.stride_tricks.sliding_window_view` gives you every length-`w` window of a signal as rows of one array — **sharing memory**, no copy. Because one input element appears in several windows, NumPy makes the result **read-only** so an accidental write cannot corrupt overlapping windows.

**Predict** the shape of the windows for a length-8 signal with `window_shape=3`.

In [5]:
signal = np.arange(8, dtype=np.float32)
windows = np.lib.stride_tricks.sliding_window_view(signal, window_shape=3)

print("signal :", signal)
print("windows:\n", windows)
print("shape  :", windows.shape, "  <- 6 windows of length 3")
print("shares memory with signal:", np.shares_memory(signal, windows))

assert windows.shape == (6, 3)
assert np.shares_memory(signal, windows)

signal : [0. 1. 2. 3. 4. 5. 6. 7.]
windows:
 [[0. 1. 2.]
 [1. 2. 3.]
 [2. 3. 4.]
 [3. 4. 5.]
 [4. 5. 6.]
 [5. 6. 7.]]
shape  : (6, 3)   <- 6 windows of length 3
shares memory with signal: True


In [6]:
print("windows.flags.writeable:", windows.flags.writeable)
try:
    windows[0, 0] = -1
except ValueError:
    print("OK: writing to an overlapping window view is refused")
else:
    raise AssertionError("An overlapping window should be read-only")

windows.flags.writeable: False
OK: writing to an overlapping window view is refused


### The unsafe version: `as_strided`

`sliding_window_view` is a safe wrapper around `np.lib.stride_tricks.as_strided`, which builds an array by writing `shape` and `strides` **with no bounds checking at all**. Wrong numbers there read whatever bytes happen to be next in memory (a silent wrong answer) or past the end of the allocation (a crash). Only use `as_strided` when you can prove every index stays inside the block — and keep the result read-only.

In [7]:
from numpy.lib.stride_tricks import as_strided

rows = signal.size - 3 + 1                # 6
step = signal.strides[0]                  # bytes between elements
manual = as_strided(signal, shape=(rows, 3), strides=(step, step), writeable=False)
safe = np.lib.stride_tricks.sliding_window_view(signal, 3)

print("manual matches the safe helper:", np.array_equal(manual, safe))
print()
print("DANGER (do not run): as_strided(signal, shape=(signal.size, 3), strides=(step, step))")
print("  would ask for 8 windows from a signal that only has room for 6 -> reads past the buffer.")

assert np.array_equal(manual, safe)

manual matches the safe helper: True

DANGER (do not run): as_strided(signal, shape=(signal.size, 3), strides=(step, step))
  would ask for 8 windows from a signal that only has room for 6 -> reads past the buffer.


## 5. Vectorization and broadcasting

- **Vectorization** — write `a + b`, not a Python `for` loop. NumPy runs the loop in fast compiled code.
- **Broadcasting** — NumPy stretches a size-1 axis to match, *without* copying the data. Handy, but a wrong shape can broadcast into a fast **wrong** answer, so check shapes on purpose.

**Predict.** `embeddings` is `(2, 3)`. With `keepdims=True`, what shape is `feature_mean`? After subtracting it, what is `centered.mean(axis=0)`?

In [8]:
embeddings = np.array([[1., 2., 3.], [4., 5., 6.]])
feature_mean = embeddings.mean(axis=0, keepdims=True)      # mean of each column
centered = embeddings - feature_mean                        # (2,3) - (1,3) broadcasts

print("feature_mean:", feature_mean, " shape", feature_mean.shape)
print("centered:\n", centered)
print("centered.mean(axis=0):", centered.mean(axis=0), " <- each column now averages to 0")

assert feature_mean.shape == (1, 3)
assert centered.shape == (2, 3)
assert np.allclose(centered.mean(axis=0), 0)

feature_mean: [[2.5 3.5 4.5]]  shape (1, 3)
centered:
 [[-1.5 -1.5 -1.5]
 [ 1.5  1.5  1.5]]
centered.mean(axis=0): [0. 0. 0.]  <- each column now averages to 0


In [9]:
# When shapes do NOT line up, broadcasting refuses (this is the good case).
try:
    embeddings + np.ones((2, 2))           # (2,3) vs (2,2) -> no
except ValueError as exc:
    print("OK, refused:", exc)
else:
    raise AssertionError("Expected a broadcasting error")

OK, refused: operands could not be broadcast together with shapes (2,3) (2,2) 


### What you just saw

Broadcasting lines up shapes from the **right**. Two axes fit if they are equal, or if one of them is `1`. `(2, 3)` and `(1, 3)` fit, so `feature_mean` is subtracted from every row with no copy. `(2, 3)` and `(2, 2)` do not fit, so NumPy raises instead of guessing.

## 6. A fixed-size ring buffer

A streaming system should not keep growing an array with `np.concatenate` — that reallocates and copies every time. Instead: allocate once, and overwrite the oldest slot when a new sample arrives. That is a **ring buffer**.

- `data` is allocated once and never resized.
- `next_index` points at the slot to overwrite next.
- `size` counts how many slots hold real data so far.
- `chronological()` returns the samples oldest-to-newest, as a **copy** so the caller cannot corrupt the buffer.

**Predict.** `RingBuffer(3, 2)` then append `[0,0], [1,-1], [2,-2], [3,-3], [4,-4]`. What does `chronological()` return?

In [10]:
class RingBuffer:
    def __init__(self, capacity: int, width: int, dtype=np.float32):
        if capacity <= 0 or width <= 0:
            raise ValueError("capacity and width must be positive")
        self.data = np.empty((capacity, width), dtype=dtype)     # allocated ONCE
        self.capacity, self.width = capacity, width
        self.next_index = self.size = 0

    def append(self, chunk: np.ndarray) -> None:
        chunk = np.asarray(chunk, dtype=self.data.dtype)
        if chunk.shape != (self.width,):
            raise ValueError(f"expected shape {(self.width,)}, got {chunk.shape}")
        self.data[self.next_index] = chunk
        self.next_index = (self.next_index + 1) % self.capacity   # wrap around
        self.size = min(self.size + 1, self.capacity)

    def chronological(self) -> np.ndarray:
        if self.size < self.capacity:
            return self.data[:self.size].copy()
        return np.concatenate((self.data[self.next_index:], self.data[:self.next_index]))

rb = RingBuffer(3, 2)
for i in range(5):
    rb.append(np.array([i, -i]))

print("after 5 appends, capacity 3, chronological():")
print(rb.chronological(), " <- only the last 3, oldest first")

assert rb.chronological().tolist() == [[2, -2], [3, -3], [4, -4]]

after 5 appends, capacity 3, chronological():
[[ 2. -2.]
 [ 3. -3.]
 [ 4. -4.]]  <- only the last 3, oldest first


In [11]:
small = RingBuffer(capacity=2, width=1)
small.append([10])

latest = small.chronological()      # this is a COPY
latest[0, 0] = -1                   # ...so scribbling on it
print("buffer after writing to the returned array:", small.chronological().tolist(),
      " <- still 10.0, the buffer is safe")
assert small.chronological().tolist() == [[10.0]]

try:
    small.append([20, 30])          # wrong width
except ValueError as exc:
    print("wrong chunk shape refused:", exc)
else:
    raise AssertionError("Expected a shape validation error")

buffer after writing to the returned array: [[10.0]]  <- still 10.0, the buffer is safe
wrong chunk shape refused: expected shape (1,), got (2,)


### What you just saw

The buffer allocated exactly `3 × 2` numbers and never grew, even though we appended 5 chunks. After the buffer fills, `chronological()` stitches the two pieces (`data[next_index:]` then `data[:next_index]`) into oldest-to-newest order.

It returns a **copy** on purpose: the second cell wrote `-1` into the returned array and the buffer was untouched. Whether an accessor returns a copy or a view is part of the API — say which, and test it.

## Project — Streaming feature extractor

Build a ring-buffer pipeline that consumes sensor chunks, exposes the latest fixed-width window, normalizes each feature, and benchmarks it against repeated concatenation.

**Concepts to test:**

- A basic slice shares memory; advanced indexing makes a copy.
- A transpose can be non-contiguous while still sharing memory.
- A sliding-window view overlaps source elements and must not be modified.
- Broadcasting must produce the intended shape, not merely a result that runs.
- A buffer with capacity `N` keeps only the latest `N` samples and preserves chronological order after wrap-around.
- The returned window has documented ownership semantics and does not unexpectedly mutate internal state.
- Invalid capacities, widths and chunk shapes raise clear errors.
- The normalized output agrees numerically with a simple reference implementation.

**Acceptance criteria:** constant allocated capacity; correct wrap-around; clear ownership semantics; shape validation; numerical agreement with a simple reference implementation.

You may `from course_utils import RingBuffer` instead of copying the class above; the module ships the same implementation with docstrings.

**Checks to run yourself**

- After many `push` calls, assert `extractor.data.nbytes` (or `.size`) never changes.
- Fill past capacity, then confirm `chronological()` is exactly the last `N` chunks in order.
- Mutate the array returned by your latest-window accessor and confirm internal state is untouched.
- Compare `normalized()` against `(w - w.mean(0)) / w.std(0)` computed with plain NumPy on the same window.
- Benchmark `push` in a loop against `np.concatenate` growth for ~100k chunks with `time.perf_counter`.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse RingBuffer from this notebook as the storage layer, or import it:
#     from course_utils import RingBuffer
#
# 1. StreamingFeatureExtractor(capacity, width): validate the arguments.
# 2. push(chunk): validate shape, store in the ring buffer.
# 3. latest_window(): return the chronological window with DOCUMENTED ownership
#    (a copy or a read-only view — state which, and test it).
# 4. normalized(): per-feature (x - mean) / std over the current window.
# 5. A benchmark of push-in-a-loop vs repeated np.concatenate growth.

import numpy as np


class StreamingFeatureExtractor:
    ...


raise NotImplementedError("Implement the streaming feature extractor")
